In [14]:
import mne
import numpy as np
import plotly.express as px
import plotly.io as pio
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
pio.renderers.default = 'notebook'
from crosci.biomarkers import (
    DFA,
    fEI
)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'code'))
from compute_criticality import load_all, criticality_dir_for

In [15]:
df = load_all(criticality_dir_for(''))
df = df[~df['bad']]
adf = df.groupby(['task', 'band', 'channel'])[['dfa', 'fei']].aggregate('mean').reset_index()

In [16]:
dataset = "ds003969"
bids_root = Path('/home/jackhoyle') / 'mne_data' / dataset
subject_info = pd.read_csv(bids_root / 'participants.tsv', sep='\t')
subject_info['subject'] = subject_info['participant_id'].str.removeprefix('sub-')
df = df.merge(subject_info[['subject', 'group']], on='subject', how='left', validate='many_to_one')

In [18]:
bands = ['alpha', 'beta']

In [20]:
df['group'].unique()

<StringArray>
['htr', 'ctr', 'tm', 'vip', 'sny']
Length: 5, dtype: str

In [25]:
# H1: alpha and beta LRTCs increase under meditation for the meditator groups
# exclude TM too because the original paper does for everything too and it's noisy
for band in bands:
    bdf = df[(df['band'] == band) & (~df['group'].isin(['tm', 'ctr']))].groupby(
        ['task', 'subject', 'group'])[['dfa']].aggregate('mean').reset_index()

    bdf['med_task'] = bdf['task'].isin(['med1breath', 'med2'])

    lm = ols('dfa ~ C(med_task)', data=bdf).fit(cov_type="HC3")
    
    print(f"Band: {band}")
    print(lm.summary())

Band: alpha
                            OLS Regression Results                            
Dep. Variable:                    dfa   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.006
Method:                 Least Squares   F-statistic:                   0.08890
Date:                Thu, 08 Oct 2026   Prob (F-statistic):              0.766
Time:                        08:52:12   Log-Likelihood:                 114.12
No. Observations:                 161   AIC:                            -224.2
Df Residuals:                     159   BIC:                            -218.1
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept             

In [33]:
# H2: alpha and beta LRTCs in meditators decrease 
# as meditation continues as there is increased meditative depth

for band in df['band'].unique():
    bdf = df[(df['band'] == band) & (~df['group'].isin(['tm', 'ctr']))].groupby(
        ['task', 'subject', 'group'])[['dfa', 'fei']].aggregate('mean').reset_index()

    lm = ols('dfa ~ C(task)', data=bdf).fit(cov_type="HC3")
    
    print(f"Band: {band}")
    print(lm.summary())

Band: delta
                            OLS Regression Results                            
Dep. Variable:                    dfa   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                 -0.008
Method:                 Least Squares   F-statistic:                    0.6119
Date:                Thu, 08 Oct 2026   Prob (F-statistic):              0.608
Time:                        09:04:49   Log-Likelihood:                 166.44
No. Observations:                 161   AIC:                            -324.9
Df Residuals:                     157   BIC:                            -312.6
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.64

In [34]:
# H2: In resting/non-meditative state there is no significant difference in alpha/beta LRTCs. 

for band in df['band'].unique():
    bdf = df[(df['band'] == band) & (~df['group'].isin(['tm']))& (~df['task'].isin(['med2', 'med1breath']))].groupby(
        ['task', 'subject', 'group'])[['dfa', 'fei']].aggregate('mean').reset_index()

    lm = ols('dfa ~ C(group)', data=bdf).fit(cov_type="HC3")
    
    print(f"Band: {band}")
    print(lm.summary())

Band: delta
                            OLS Regression Results                            
Dep. Variable:                    dfa   R-squared:                       0.052
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     2.960
Date:                Thu, 08 Oct 2026   Prob (F-statistic):             0.0364
Time:                        09:06:41   Log-Likelihood:                 92.025
No. Observations:                  96   AIC:                            -176.1
Df Residuals:                      92   BIC:                            -165.8
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.6547    

In [ ]:
# differences by task?

for band in df['band'].unique():
    bdf = df[df['band'] == band].groupby(['task', 'subject'])[['dfa', 'fei']].aggregate('mean').reset_index()

    lm = ols('dfa ~ C(task)', data=bdf).fit(cov_type="HC3")

    print(f'Band: {band}')
    print(lm.summary())

    print()


In [ ]:
# differences by task?

for band in df['band'].unique():
    bdf = df[df['band'] == band].groupby(['task', 'subject'])[['dfa', 'fei']].aggregate('mean').reset_index()

    lm = ols('fei ~ C(task)', data=bdf).fit(cov_type="HC3")

    print(f'Band: {band}')
    print(lm.summary())

    print()


In [ ]:
# differences by task and group?

for band in df['band'].unique():
    bdf = df[df['band'] == band].groupby(['task', 'subject', 'group'])[['dfa', 'fei']].aggregate('mean').reset_index()

    lm = ols('dfa ~ C(task) * C(group)', data=bdf).fit(cov_type="HC3")

    print(f'Band: {band}')
    print(lm.summary())

    print()


In [ ]:
df